# Extraction 


In [1]:
#import neccassary packages 
from pydantic import BaseModel, Field, field_validator
from datetime import date
import sys
import os
import fitz
import time
from google.genai import errors
from google import genai
from google.genai import types
from pydantic import ValidationError
from dotenv import load_dotenv
import mysql.connector
from pathlib import Path

In [2]:
class GradeBand(BaseModel):
    band_name: str | None = Field(
        default=None,
        description="The band heading exactly as written, e.g. 'Excellent', 'Very Good', 'Limited/No Evidence'."
    )
    descriptor: str | None = Field(
        default=None,
        description="The text in this band's cell for this criterion, as written in the brief. Null if the cell is empty."
    )


class Criterion(BaseModel):
    name: str | None = Field(
        default=None,
        description="Name of the marking criterion (the row heading), as written in the brief."
    )
    description: str | None = Field(
        default=None,
        description="A general description of what the criterion assesses, only if the brief gives one separately from the grade band descriptors. Null otherwise."
    )
    weighting_percent: float | None = Field(
        default=None,
        description="Weighting of this criterion as a number only, e.g. 30 for '30%'. Null if not stated."
    )
    grade_bands: list[GradeBand] = Field(
        default_factory=list,
        description="One entry per grade band for this criterion. Empty list if the brief gives no band descriptors."
    )


class AssessmentBrief(BaseModel):
    programme: str | None = Field(
        default=None,
        description="Name of the programme or course the module belongs to. Null if not stated."
    )
    module_code: str | None = Field(
        default=None,
        description="Module code exactly as written, e.g. 'CS4001'."
    )
    module_title: str | None = Field(
        default=None,
        description="Full title of the module."
    )
    module_level: int | None = Field(
        default=None,
        description="Academic level of the module as a number, e.g. 4, 5, 6 or 7. Null if not stated."
    )

    assessment_type: str | None = Field(
        default=None,
        description="Type of assessment as written, e.g. 'Essay', 'Report', 'Presentation', 'Exam'."
    )
    weighting_percent: float | None = Field(
        default=None,
        description="How much this assessment contributes to the module mark, as a number only, e.g. 100 for '100%'. Null if not stated."
    )

    word_count: int | None = Field(
        default=None,
        description="Maximum word count as a number only. Null if the assessment is not word-limited."
    )
    duration_minutes: int | None = Field(
        default=None,
        description="Duration in minutes for timed assessments such as exams or presentations. Null if not timed."
    )

    deadline: date | None = Field(
        default=None,
        description="Submission deadline as an ISO date (YYYY-MM-DD). Null if only a week number or vague wording is given."
    )
    deadline_raw: str | None = Field(
        default=None,
        description="The submission deadline exactly as written in the document, e.g. 'Week 10, Monday 12pm'."
    )
    feedback_date: date | None = Field(
        default=None,
        description="Date feedback is returned to students as an ISO date (YYYY-MM-DD). Null if not a specific date."
    )
    feedback_date_raw: str | None = Field(
        default=None,
        description="The feedback return date exactly as written in the document."
    )

    submission_format: str | None = Field(
        default=None,
        description="How and in what format the work is submitted, e.g. 'PDF via Turnitin'. Null if not stated."
    )

    learning_outcomes: list[str] = Field(
        default_factory=list,
        description="Each learning outcome assessed, as a separate string. Empty list if none are stated."
    )
    marking_criteria: list[Criterion] = Field(
        default_factory=list,
        description="Each marking criterion with its grade band descriptors. Empty list if none are stated."
    )
    requirements: list[str] = Field(
        default_factory=list,
        description="Explicit requirements the submission must meet, e.g. referencing style or sections that must be included. One per string."
    )
    success_guidance: str | None = Field(
        default=None,
        description=(
            "The brief's guidance on how to do well or succeed in this assessment, "
            "copied as written. Do not summarise. Null if there is no such section."
        ),
    )
    layout_style_guidance: str | None = Field(
        default=None,
        description=(
            "Guidance on layout, structure, formatting, tone or writing style "
            "(e.g. headings, fonts, line spacing, report structure), copied as written. Null if not stated."
        ),
    )
    referencing: str | None = Field(
        default=None,
        description=(
            "Referencing and citation requirements, e.g. the referencing style to use "
            "and any rules about sources, copied as written. Null if not stated."
        ),
    )
    employability_professional_skills: str | None = Field(
        default=None,
        description=(
            "Any section describing employability, professional or transferable skills "
            "this assessment develops, copied as written. Null if not stated."
        ),
    )
    ai_use: str | None = Field(
        default=None,
        description=(
            "The brief's rules or guidance on the use of generative AI, "
            "copied exactly as written. Null if not stated."
        ),
    )
    ethical_requirements: str | None = Field(
        default=None,
        description=(
            "Ethical requirements or approvals relevant to the assessment "
            "(e.g. consent, data protection, research ethics), copied as written. Null if not stated."
        ),
    )
    submission_checklist: list[str] | None = Field(
        default=None,
        description=(
            "The submission checklist or final-check list, copied as written, "
            "keeping each item on its own line. Null if not stated."
        ),
    )
    task_description: str | None = Field(
        default=None,
        description=(
            "The full text of the assessment task, i.e. what students are asked to do, "
            "copied exactly as written in the brief. Do not summarise or reword. "
            "Include any bullet points or numbered parts. Do not include the marking "
            "criteria, learning outcomes or deadlines. Null if not stated."
        ),
    )
    @field_validator("word_count", "duration_minutes")
    @classmethod
    def must_be_positive(cls, value):
        if value is not None and value <= 0:
            raise ValueError("Value must be positive")
        return value

In [3]:
MODEL = "gemini-3.8-flash"

In [4]:
load_dotenv()

def extract_text(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    pages = []
    for i, page in enumerate(doc, start=1):
        pages.append(f"[PAGE {i}]\n{page.get_text()}")
    return "\n\n".join(pages)

def extract_brief(text, max_attempts=6):
    client = genai.Client()
    prompt = (
        "Extract the assessment brief details from the document below. "
        "Only use information stated in the document. "
        "Use null for anything not stated. Do not guess.\n\n"
        f"DOCUMENT:\n{text}"
    )

    for attempt in range(max_attempts):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=AssessmentBrief,
                ),
            )
            return AssessmentBrief.model_validate_json(response.text)
        except errors.ServerError as e:
            wait = 2 ** attempt * 5   # 5, 10, 20, 40, 80, 160 seconds
            print(f"Server error {e.code}, retrying in {wait}s (attempt {attempt + 1}/{max_attempts})")
            time.sleep(wait)

    raise RuntimeError("Gemini still unavailable after several attempts")
    
if __name__ == "__main__":
    pdf_path = "sample_pdf.pdf"
    text = extract_text(pdf_path)

    print("=== RAW TEXT (first 1500 chars) ===")
    print(text[:1500])

    try:
        brief = extract_brief(text)
    except ValidationError as e:
        print("\n=== VALIDATION FAILED ===")
        print(e)
        sys.exit(1)

    print("\n=== EXTRACTED ===")
    print(brief.model_dump_json(indent=2))


=== RAW TEXT (first 1500 chars) ===
[PAGE 1]
ASSESSMENT BRIEF  
  
Programme  
All BSc Computing Programmes 
Module Title  
Database 
Module Code  
CMP020C106S 
Module Level  
4 
Assessment Type(s)  
 Report 
Word Length / Duration  
 None 
% contribution to 
module mark  
 %40 
Deadline (date & time) for 
Submission  
4th of March 
2026 
Format/Location of 
submission  
 Moodle Report 
Submission 
Assessment Feedback 
date:  
1st of April, 
2026 
  
  
Learning Outcomes  
LO1: Practise business database design 
Employability and Professional Skills   
This assessment has been designed to 
provide you with an opportunity to 
demonstrate your achievement of the 
following skills, which are critical in 
professional contexts and employment 
settings. 
By successfully completing this 
assessment, you will be able to: 
• Apply analytical and problem-solving 
skills to translate business 
requirements into effective database 
designs. 
• Use equivalent industry-standard 
tools and technique

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



=== EXTRACTED ===
{
  "programme": "All BSc Computing Programmes",
  "module_code": "CMP020C106S",
  "module_title": "Database",
  "module_level": 4,
  "assessment_type": "Report",
  "weighting_percent": 40.0,
  "word_count": null,
  "duration_minutes": null,
  "deadline": "2026-03-04",
  "deadline_raw": "4th of March 2026",
  "feedback_date": "2026-04-01",
  "feedback_date_raw": "1st of April, 2026",
  "submission_format": "Moodle Report Submission",
  "learning_outcomes": [
    "LO1: Practise business database design"
  ],
  "marking_criteria": [
    {
      "name": "Idea",
      "description": "the suitability of the idea for a RDBMS",
      "weighting_percent": null,
      "grade_bands": [
        {
          "band_name": "Excellent",
          "descriptor": "The idea requires a RDBMS and has been well specified."
        },
        {
          "band_name": "Satisfactory",
          "descriptor": "The idea would require a RDBMS to manage although the idea is not fully specified."


In [5]:
def extract_images(pdf_path, out_root="extracted_images", min_size=100):
    """Save meaningful images from a PDF and return a record of each one."""
    pdf_path = Path(pdf_path)
    out_dir = Path(out_root) / pdf_path.stem
    out_dir.mkdir(parents=True, exist_ok=True)

    doc = fitz.open(pdf_path)
    seen_xrefs = set()
    results = []

    for page_index in range(len(doc)):
        page = doc[page_index]
        page_number = page_index + 1

        for image_index, img in enumerate(page.get_images(full=True), start=1):
            xref = img[0]
            if xref in seen_xrefs:
                continue  # same image repeated (e.g. logo in a header)
            seen_xrefs.add(xref)

            base = doc.extract_image(xref)
            if base["width"] < min_size or base["height"] < min_size:
                continue  # probably an icon or decoration

            image_path = out_dir / f"page{page_number}_img{image_index}.{base['ext']}"
            image_path.write_bytes(base["image"])

            results.append({
                "source_file": pdf_path.name,
                "page_number": page_number,
                "image_path": str(image_path),
                "width": base["width"],
                "height": base["height"],
            })

    print(f"{pdf_path.name}: saved {len(results)} images")
    return results

In [6]:
MIME = {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg", "webp": "image/webp"}

def describe_image(image_path):
    ext = Path(image_path).suffix.lstrip(".").lower()
    if ext not in MIME:
        return None  # unsupported format, skip for now

    client = genai.Client()
    prompt = (
        "This image comes from a university assessment brief. Describe only what is visible. "
        "Transcribe any text in it. If it is an example, diagram or screenshot, "
        "explain what it shows. If it is purely decorative (logo, banner, icon), "
        "reply with exactly: DECORATIVE"
    )
    response = client.models.generate_content(
        model=MODEL,
        contents=[
            types.Part.from_bytes(data=Path(image_path).read_bytes(), mime_type=MIME[ext]),
            prompt,
        ],
    )
    return response.text.strip()

In [7]:
load_dotenv()

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=os.environ["MYSQL_PASSWORD"],
    database="rag_system",
)

print("Connected successfully!")

conn.close()

Connected successfully!


In [8]:
def get_connection():
    return mysql.connector.connect(
        host="localhost",
        user="root",  # or your own MySQL user
        password=os.environ["MYSQL_PASSWORD"],  # don't hard-code it
        database="rag_system",
    )

In [9]:
def save_brief(brief, source_file):
    data = {
        "source_file": source_file,
        **brief.model_dump(exclude={"learning_outcomes", "marking_criteria", "requirements","submission_checklist"}),
    }

    columns = ", ".join(data.keys())
    placeholders = ", ".join(["%s"] * len(data))
    updates = ", ".join(
        f"{col}=VALUES({col})" for col in data if col != "source_file"
    )

    sql = f"""
        INSERT INTO assessments ({columns})
        VALUES ({placeholders})
        ON DUPLICATE KEY UPDATE {updates}
    """

    conn = get_connection()
    try:
        cur = conn.cursor()
        cur.execute(sql, list(data.values()))
        conn.commit()
    finally:
        conn.close()

In [10]:
def save_list_items(table, assessment_id, items):
    conn = get_connection()
    try:
        cur = conn.cursor()
        cur.execute(f"DELETE FROM {table} WHERE assessment_id = %s", (assessment_id,))
        for item in items:
            cur.execute(
                f"INSERT INTO {table} (assessment_id, text) VALUES (%s, %s)",
                (assessment_id, item),
            )
        conn.commit()
    finally:
        conn.close()

In [11]:
def get_assessment_id(source_file):
    conn = get_connection()
    try:
        cur = conn.cursor()
        cur.execute("SELECT id FROM assessments WHERE source_file = %s", (source_file,))
        row = cur.fetchone()
        if row is None:
            raise ValueError(f"No assessment saved for {source_file}")
        return row[0]
    finally:
        conn.close()

In [12]:
def save_rubric(brief, source_file):
    conn = get_connection()
    try:
        cur = conn.cursor()
        cur.execute("SELECT id FROM assessments WHERE source_file = %s", (source_file,))
        assessment_id = cur.fetchone()[0]

        # clear old rubric so re-running doesn't duplicate rows
        cur.execute("DELETE FROM criteria WHERE assessment_id = %s", (assessment_id,))

        for c in brief.marking_criteria:
            cur.execute(
                "INSERT INTO criteria (assessment_id, name, description, weighting_percent) VALUES (%s, %s, %s, %s)",
                (assessment_id, c.name, c.description, c.weighting_percent),
            )
            criterion_id = cur.lastrowid
            for b in c.grade_bands:
                cur.execute(
                    "INSERT INTO grade_bands (criterion_id, band_name, descriptor) VALUES (%s, %s, %s)",
                    (criterion_id, b.band_name, b.descriptor),
                )
        conn.commit()
    finally:
        conn.close()

In [13]:
MIME = {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg", "webp": "image/webp"}

def describe_image(image_path):
    ext = Path(image_path).suffix.lstrip(".").lower()
    if ext not in MIME:
        return None  # unsupported format, skip for now

    client = genai.Client()
    prompt = (
        "This image comes from a university assessment brief. Describe only what is visible. "
        "Transcribe any text in it. If it is an example, diagram or screenshot, "
        "explain what it shows. If it is purely decorative (logo, banner, icon), "
        "reply with exactly: DECORATIVE"
    )
    response = client.models.generate_content(
        model=MODEL,
        contents=[
            types.Part.from_bytes(data=Path(image_path).read_bytes(), mime_type=MIME[ext]),
            prompt,
        ],
    )
    return response.text.strip()

In [14]:
def save_images(source_file, image_records):
    assessment_id = get_assessment_id(source_file)  # same helper you already have
    conn = get_connection()
    try:
        cur = conn.cursor()
        cur.execute("DELETE FROM task_images WHERE assessment_id = %s", (assessment_id,))
        for rec in image_records:
            description = describe_image(rec["image_path"])
            if description is None or description == "DECORATIVE":
                continue
            cur.execute(
                "INSERT INTO task_images (assessment_id, page_number, image_path, description) "
                "VALUES (%s, %s, %s, %s)",
                (assessment_id, rec["page_number"], rec["image_path"], description),
            )
        conn.commit()
    finally:
        conn.close()

In [15]:
brief = extract_brief(text)

In [16]:
save_brief(brief, "my_brief.pdf")    

In [17]:
cur_id = get_assessment_id("my_brief.pdf") 

In [18]:
save_list_items("learning_outcomes", cur_id, brief.learning_outcomes)

In [19]:
save_rubric(brief, "my_brief.pdf")

In [21]:
pdf_path = "sample_pdf.pdf"
filename = "my_brief.pdf" 
images = extract_images(pdf_path)
save_images(filename, images)

sample_pdf.pdf: saved 2 images


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}